# League of Legends ELO System - Data Cleaning and Name Resolution
First, we'll import our libraries and load the Oracle's Elixir datasets for 2024 and 2025 to inspect player names, teams, and leagues. Our goal here is to identify and resolve fuzzy name errors before building the ELO engine.

In [1]:
import pandas as pd
import numpy as np
import os
import glob

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

## Load Datasets
We will load and combine 2024 and 2025 data to find inconsistencies across years.

In [2]:
data_dir = '../data/csv/'
files = [f for f in glob.glob(os.path.join(data_dir, '*2024*.csv')) + glob.glob(os.path.join(data_dir, '*2025*.csv')) if not f.endswith('.bak')]

dfs = []
for f in files:
    try:
        df = pd.read_csv(f, low_memory=False)
        dfs.append(df)
        print(f"Loaded {f}: {df.shape[0]} rows")
    except Exception as e:
        print(f"Error loading {f}: {e}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    # Oracle's Elixir has 'playername', 'teamname', 'league'
    print(f"\nTotal Combined Rows: {df_all.shape[0]}")
else:
    print("No datasets loaded. Please check the path.")

Loaded ../data/csv\2024_LoL_esports_match_data_from_OraclesElixir.csv: 122388 rows
Loaded ../data/csv\2025_LoL_esports_match_data_from_OraclesElixir.csv: 120636 rows

Total Combined Rows: 243024


In [3]:
# We want to identify inconsistencies in player and team naming.
# First, let's look at the basic info, then count unique players per team.
players_only = df_all[df_all['position'] != 'team']

print("Total matches:", len(df_all['gameid'].unique()))
print("Total unique player names:", len(players_only['playername'].dropna().unique()))
print("Total unique teams:", len(df_all['teamname'].dropna().unique()))

# Example: Do any players play for multiple teams in the dataset?
player_team_counts = players_only.groupby('playername')['teamname'].nunique().sort_values(ascending=False)
print("\nPlayers who played for the most teams in 24/25:")
print(player_team_counts.head(10))

# Are there any players with the exact same name? (We can sometimes check this by using player ID if Oracles provides one)
if 'playerid' in df_all.columns:
    print(f"\nUnique player IDs: {len(players_only['playerid'].dropna().unique())}")
else:
    print("\nNo 'playerid' column found. We will rely purely on 'playername'.")
    
# Let's inspect rows with missing player names to see what they are
print("\nRows with missing player names:")
missing_players = players_only[players_only['playername'].isna()]
print(f"Count: {len(missing_players)}")

Total matches: 20252
Total unique player names: 4206
Total unique teams: 712

Players who played for the most teams in 24/25:
playername
Seal           8
RustySniper    8
FSZ            7
Faded          7
Knighter       7
Black          7
PatxiPirata    7
Kaze           7
Rewound        6
Neramin        6
Name: teamname, dtype: int64

Unique player IDs: 3965

Rows with missing player names:
Count: 0


In [4]:
# Let's map out name changes where multiple playernames share one playerid
id_to_names = players_only.groupby('playerid')['playername'].unique()
name_changes = id_to_names[id_to_names.apply(len) > 1]
print(f"Players with multiple names (name changes): {len(name_changes)}")
for pid, names in name_changes.head(10).items():
    print(f"{pid}: {names}")

# Now same for teams
if 'teamid' in df_all.columns:
    print(f"\nUnique team IDs: {len(df_all['teamid'].dropna().unique())}")
    team_id_to_names = df_all.groupby('teamid')['teamname'].unique()
    team_name_changes = team_id_to_names[team_id_to_names.apply(len) > 1]
    print(f"Teams with multiple names: {len(team_name_changes)}")
    for tid, names in team_name_changes.head(10).items():
        print(f"{tid}: {names}")

Players with multiple names (name changes): 0

Unique team IDs: 596
Teams with multiple names: 0


In [5]:
# Wait, earlier we found 4206 unique names but only 3965 unique IDs. 
# And 0 IDs have multiple names. This means some NAMES correspond to MULTIPLE IDs.
# Let's confirm: Different people with the exact same gamertag.
name_to_ids = players_only.groupby('playername')['playerid'].unique()
shared_names = name_to_ids[name_to_ids.apply(len) > 1]
print(f"Names used by multiple distinct player IDs: {len(shared_names)}")
for name, ids in list(shared_names.items())[:10]:
    print(f"[{name}] maps to IDs: {ids}")

Names used by multiple distinct player IDs: 45
[Abyss] maps to IDs: <StringArray>
['oe:player:41e2980a2f1bcdbc20910125be7f939', 'oe:player:0997237547f63b0119f6816c13377e9', 'oe:player:41105719857686d3d4cdd7356d23b0a']
Length: 3, dtype: str
[Ace] maps to IDs: <StringArray>
['oe:player:73933848061d300bbe7293d9b38a2d4', 'oe:player:83352405b08076a5c04364b203e709e']
Length: 2, dtype: str
[Aegis] maps to IDs: <StringArray>
['oe:player:778c6c3733d919a3b64885912be15a9', 'oe:player:958eb9ed914a4b987237b117c8cb6f4']
Length: 2, dtype: str
[Azura] maps to IDs: <StringArray>
['oe:player:1d8e6ad53056c022bad5faeb9fdaa01', 'oe:player:6a442dcf7f93e19699e2e657a2e0eec']
Length: 2, dtype: str
[Black] maps to IDs: <StringArray>
['oe:player:dbc932ea3d30b98a49d66e17c6c4d62', 'oe:player:1f05b5fb18e8b6376f8781cc1b517b2']
Length: 2, dtype: str
[Blue] maps to IDs: <StringArray>
['oe:player:2a4f745da387ba8d9546e89e19ff74d', 'oe:player:b31e35906373123acf6483a837e966c']
Length: 2, dtype: str
[Bruno] maps to IDs: <S

In [6]:
# Let's inspect the leagues available so we can establish our SOS / Tier System 
# for regional starting ELOs and regional modifiers.
print("Total unique leagues:", len(df_all['league'].dropna().unique()))
league_counts = df_all[['gameid', 'league']].drop_duplicates()['league'].value_counts()
print("\nLeagues by number of matches played (Top 20):")
print(league_counts.head(20))

# Can we identify international tournaments? (MSI, Worlds typically denoted by 'WCC', 'MSI', 'Worlds')
tournaments = [l for l in league_counts.index if any(x in str(l).upper() for x in ['WCC', 'MSI', 'WORLD', 'EWC'])]
print(f"\nPotential International/Inter-League Events found: {tournaments}")

Total unique leagues: 64

Leagues by number of matches played (Top 20):
league
LPL       1522
LCKC      1053
LCK       1037
EM         839
LAS        778
NACL       764
LEC        600
PCS        600
LDL        565
LFL        555
PRM        544
LJL        522
LVP SL     506
NLC        457
AL         446
TCL        421
LFL2       405
VCS        393
HM         345
PCL        340
Name: count, dtype: int64

Potential International/Inter-League Events found: ['MSI', 'EWC']


In [7]:
# Let's test our new ELO Engine and run an initial "Burn-In" pass over the Q1 2024 data.
import sys
sys.path.append('../src')
from features.elo import PlayerEloSystem
from datetime import datetime

# Initialize engine
elo_engine = PlayerEloSystem()

# Ensure chronological order and proper date parsing
df_all['date'] = pd.to_datetime(df_all['date'])
df_all_sorted = df_all.sort_values(by='date').reset_index(drop=True)

# Separate into games and players
match_results = df_all_sorted[df_all_sorted['position'] == 'team'][['gameid', 'date', 'league', 'teamid', 'teamname', 'result', 'side']]
players_df = df_all_sorted[df_all_sorted['position'] != 'team'][['gameid', 'teamid', 'playerid', 'playername']]

print(f"Total Matches to Process: {len(match_results['gameid'].unique())}")

Total Matches to Process: 20252


In [ ]:
# We will process the first 1000 matches to ensure the engine is humming nicely
# Grouping the players by gameid and teamid
grouped_players = players_df.groupby(['gameid', 'teamid'])['playerid'].apply(list).reset_index()

game_records = []
engine_stats = []

for i, game_id in enumerate(match_results['gameid'].unique()[:1000]):
    game_data = match_results[match_results['gameid'] == game_id]
    if len(game_data) != 2:
        continue # skip incomplete data
    
    # Extract date, league from any team row
    date = pd.to_datetime(game_data['date'].iloc[0])
    league = game_data['league'].iloc[0]
    
    team_a = game_data.iloc[0]
    team_b = game_data.iloc[1]
    
    team_a_won = bool(team_a['result'])
    
    # Find players from grouped dataframe
    team_a_players = grouped_players[(grouped_players['gameid'] == game_id) & (grouped_players['teamid'] == team_a['teamid'])]['playerid']
    team_b_players = grouped_players[(grouped_players['gameid'] == game_id) & (grouped_players['teamid'] == team_b['teamid'])]['playerid']
    
    if len(team_a_players) == 0 or len(team_b_players) == 0:
        continue
    
    players_a = list(team_a_players.values[0])
    players_b = list(team_b_players.values[0])
    
    # Process through Engine
    result = elo_engine.process_match(date, league, players_a, players_b, team_a_won)
    
    engine_stats.append({
        'gameid': game_id,
        'date': date,
        'league': league,
        'team_a': team_a['teamname'],
        'team_b': team_b['teamname'],
        'team_a_elo_pre': result['avg_a_elo_pre'],
        'team_b_elo_pre': result['avg_b_elo_pre'],
        'expected_a': result['expected_a'],
        'winner': team_a['teamname'] if team_a_won else team_b['teamname']
    })
    
# Convert results to dataframe to inspect
elo_results_df = pd.DataFrame(engine_stats)
print("Processed 1000 matches successfully!")
print(elo_results_df.head())

Processed 1000 matches successfully!
               gameid                date league       team_a       team_b  team_a_elo_pre  team_b_elo_pre  expected_a       winner
0  10660-10660_game_1 2024-01-01 05:13:15   DCup    Rare Atom  LNG Esports     1350.000000     1350.000000    0.500000    Rare Atom
1  10660-10660_game_2 2024-01-01 06:03:26   DCup    Rare Atom  LNG Esports     1370.000000     1330.000000    0.557312    Rare Atom
2  10660-10660_game_3 2024-01-01 06:54:35   DCup    Rare Atom  LNG Esports     1387.707535     1319.833972    0.596454  LNG Esports
3  10660-10660_game_4 2024-01-01 07:35:57   DCup  LNG Esports    Rare Atom     1343.692132     1363.849375    0.471024    Rare Atom
4  10661-10661_game_1 2024-01-01 08:45:42   DCup  Top Esports    JD Gaming     1350.000000     1350.000000    0.500000    JD Gaming


: 